# 🛡️ PhishShield - Model 2: Deep Learning LSTM (TensorFlow / Keras)

This notebook trains and evaluates a **Bidirectional LSTM** deep neural network using **TensorFlow / Keras**:
- **Dataset:** `phishing_email.csv` (~82,486 emails)
- **Architecture:** Text Vectorization / Tokenizer -> Embedding (dim=64) -> Bidirectional LSTM (units=64) -> Dense + Dropout -> Sigmoid Output
- **Target:** Sequence-level probability prediction (0 = Legitimate, 1 = Phishing)
- **Export:** `backend/ml/saved_models/lstm_model.keras`

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, confusion_matrix, roc_curve

print(f"TensorFlow Version: {tf.__version__}")

## 1. Load Data & Text Preprocessing

In [ ]:
df = pd.read_csv("phishing_email.csv")
X_texts = df["text_combined"].astype(str).values
y_labels = df["label"].astype(int).values

print(f"Total email records: {len(df):,}")

MAX_WORDS = 20000
MAX_LEN = 120

print("Tokenizing texts with Keras Tokenizer...")
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_texts)

sequences = tokenizer.texts_to_sequences(X_texts)
padded_sequences = pad_sequences(sequences, maxlen=MAX_LEN, padding="post", truncating="post")
print("Sequence tensor shape:", padded_sequences.shape)

## 2. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, y_labels, test_size=0.20, random_state=42, stratify=y_labels
)
print(f"Train samples: {len(X_train):,}")
print(f"Test samples:  {len(X_test):,}")

## 3. Build Bidirectional LSTM Architecture in TensorFlow

In [ ]:
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.3),
    Dense(32, activation="relu"),
    Dropout(0.2),
    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## 4. Train Model

In [ ]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    epochs=3,
    batch_size=128,
    validation_split=0.15,
    callbacks=[early_stop],
    verbose=1
)

## 5. Evaluation & Metrics

In [ ]:
y_prob = model.predict(X_test, batch_size=256).flatten()
y_pred = (y_prob >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print(f"TensorFlow LSTM Accuracy: {acc * 100:.2f}%")
print(f"TensorFlow LSTM ROC-AUC : {auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Legitimate (0)", "Phishing (1)"], digits=4))

## 6. Training Loss & Accuracy Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy curve
axes[0].plot(history.history["accuracy"], label="Train Accuracy", color="royalblue")
axes[0].plot(history.history["val_accuracy"], label="Val Accuracy", color="darkorange")
axes[0].set_title("LSTM Model Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss curve
axes[1].plot(history.history["loss"], label="Train Loss", color="royalblue")
axes[1].plot(history.history["val_loss"], label="Val Loss", color="crimson")
axes[1].set_title("LSTM Binary Crossentropy Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Export TensorFlow Model & Tokenizer

In [ ]:
save_dir = os.path.join("backend", "ml", "saved_models")
os.makedirs(save_dir, exist_ok=True)

model.save(os.path.join(save_dir, "lstm_model.keras"))

tokenizer_json = tokenizer.to_json()
with open(os.path.join(save_dir, "lstm_tokenizer.json"), "w", encoding="utf-8") as f:
    f.write(tokenizer_json)

print("TensorFlow LSTM model & tokenizer saved successfully!")